# MLOps Capstone — Notebook 3: Hotel Recommendation System
## Collaborative Filtering with SVD

**Project Summary:**  
This notebook builds a hotel recommendation system using **collaborative filtering** — predicting which hotels a user would prefer based on the behaviour of similar users. The model uses **Truncated SVD (Singular Value Decomposition)** to decompose the user-hotel interaction matrix into latent factors, then reconstructs predicted ratings for unvisited hotels.

**GitHub Repository:** `https://github.com/<your-username>/mlops-travel-capstone`

---
| Section | Content |
|---------|--------|
| 0 | Install & Import |
| 1 | Load Datasets |
| 2 | EDA — Hotels & Users |
| 3 | Build User-Hotel Interaction Matrix |
| 4 | SVD Collaborative Filtering Model |
| 5 | Hyperparameter Tuning (n_components) |
| 6 | MLflow Tracking |
| 7 | Evaluation (Coverage · Diversity · Precision@K) |
| 8 | Recommendation Function |
| 9 | Save Model & Artefacts |
| 10 | MLflow Run Summary |

## 0. Install & Import Libraries

In [ ]:
!pip install mlflow scikit-learn pandas numpy matplotlib seaborn joblib scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.model_selection import KFold
import mlflow
import mlflow.sklearn

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded. MLflow version:', mlflow.__version__)

## 1. Load Datasets

In [ ]:
hotels = pd.read_csv('hotels.csv')
users  = pd.read_csv('users.csv')
print('hotels shape:', hotels.shape)
print('users  shape:', users.shape)
print('\nHotels columns:', list(hotels.columns))
display(hotels.head())
print('\nMissing values — hotels:', hotels.isnull().sum().sum())
print('Missing values — users :', users.isnull().sum().sum())

## 2. EDA — Hotels & Users

Understanding the distribution of hotel visits, spend patterns, and stay durations is essential before building a recommendation system. We also check sparsity of the user-hotel interaction matrix — a key property that determines which algorithm to use.

In [ ]:
print('=== Hotel Stats ===')
print(hotels.describe())
print('\nHotel visit counts:')
print(hotels['name'].value_counts())
print('\nHotel places:')
print(hotels['place'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Hotels EDA', fontsize=13, fontweight='bold')

# Price per night
axes[0,0].hist(hotels['price'], bins=30, color='teal', edgecolor='white')
axes[0,0].axvline(hotels['price'].mean(), color='red', linestyle='--', label=f'Mean: {hotels["price"].mean():.0f}')
axes[0,0].set_title('Price per Night Distribution'); axes[0,0].legend()

# Total spend
axes[0,1].hist(hotels['total'], bins=30, color='steelblue', edgecolor='white')
axes[0,1].set_title('Total Spend Distribution')

# Avg total spend by hotel
hotels.groupby('name')['total'].mean().sort_values().plot(kind='barh', ax=axes[0,2], color='steelblue', edgecolor='white')
axes[0,2].set_title('Avg Total Spend by Hotel')

# Days distribution
hotels['days'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], color='coral', edgecolor='white')
axes[1,0].set_title('Stay Duration Distribution'); axes[1,0].tick_params(axis='x', rotation=0)

# Bookings per hotel
hotels['name'].value_counts().plot(kind='bar', ax=axes[1,1], color='purple', edgecolor='white')
axes[1,1].set_title('Total Bookings per Hotel'); axes[1,1].tick_params(axis='x', rotation=45)

# Price by hotel
hotels.groupby('name')['price'].mean().sort_values().plot(kind='barh', ax=axes[1,2], color='teal', edgecolor='white')
axes[1,2].set_title('Avg Price per Night by Hotel')

plt.tight_layout()
plt.savefig('nb3_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Hotels visited per user
hotels_per_user = hotels.groupby('userCode')['name'].nunique()
print('Hotels visited per user:')
print(hotels_per_user.value_counts().sort_index())
print(f'\nUsers who visited all 9 hotels: {(hotels_per_user == 9).sum()}')
print(f'Users who visited < 5 hotels  : {(hotels_per_user < 5).sum()}')
print(f'\nUnique users in hotels dataset: {hotels["userCode"].nunique()}')
print(f'Users in users.csv            : {len(users)}')

## 3. Build User-Hotel Interaction Matrix

Collaborative filtering requires a **user × item matrix** where each cell represents how strongly a user has interacted with an item. We use **total spend** as the implicit rating signal — a user who spent more at a hotel implicitly preferred it over others.

- Rows: user codes
- Columns: hotel names (9 hotels)
- Values: total spend (0 = not visited)

In [ ]:
# Sum total spend per user per hotel
hotel_raw = hotels.groupby(['userCode','name'])['total'].sum().reset_index()
hotel_raw.columns = ['userCode','hotel','implicit_rating']

# Pivot to user × hotel matrix
INTERACTION_MATRIX = hotel_raw.pivot_table(
    index='userCode', columns='hotel', values='implicit_rating', fill_value=0)

print('Interaction matrix shape:', INTERACTION_MATRIX.shape)
print('Hotels:', list(INTERACTION_MATRIX.columns))
sparsity = (INTERACTION_MATRIX == 0).sum().sum() / INTERACTION_MATRIX.size * 100
print(f'Sparsity: {sparsity:.1f}% zeros')
display(INTERACTION_MATRIX.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap of first 50 users
sample_matrix = INTERACTION_MATRIX.head(50)
sns.heatmap(sample_matrix, cmap='YlOrRd', ax=axes[0], linewidths=0.1,
            cbar_kws={'label': 'Total Spend'})
axes[0].set_title('User-Hotel Interaction Matrix (first 50 users)')
axes[0].set_xlabel('Hotel'); axes[0].set_ylabel('User Code')

# Distribution of implicit ratings
ratings_flat = hotel_raw['implicit_rating']
axes[1].hist(ratings_flat, bins=40, color='teal', edgecolor='white', log=True)
axes[1].set_title('Distribution of Implicit Ratings (log scale)')
axes[1].set_xlabel('Total Spend'); axes[1].set_ylabel('Count (log)')

plt.tight_layout()
plt.savefig('nb3_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### Why collaborative filtering over content-based filtering?

Content-based filtering recommends hotels similar to what a user has visited before (based on hotel features like price, location). Collaborative filtering recommends hotels based on what *similar users* preferred — capturing patterns content-based filtering misses (e.g., users with similar spending profiles tend to prefer the same hotels even across different locations).

With 1,310 users × 9 hotels, we have enough user overlap to find meaningful similarities. Content-based would need rich hotel feature data (amenities, ratings, categories) which this dataset doesn't provide.

### Why SVD over direct cosine similarity?

Direct cosine similarity on the raw matrix compares users based on their exact spend amounts. SVD first decomposes the matrix into latent factors (hidden preference dimensions) and then computes similarity in that lower-dimensional space — filtering out noise and making the similarity measure more robust.

### Why total spend as implicit rating instead of booking count?

Booking count treats a 1-night budget stay the same as a 10-night luxury stay. Total spend better captures the *strength* of preference — a user who spent ₹5,000 at Hotel AU clearly has a stronger preference for it than one who spent ₹150.

## 4. SVD Collaborative Filtering Model

**Why SVD?**  
With only 9 hotels, the interaction matrix is small but dense. Truncated SVD decomposes it into latent factors that capture hidden patterns — e.g., users who prefer budget hotels, users who prefer long stays, etc. — without needing explicit feature engineering.

**How it works:**  
`Matrix (users × hotels)` → SVD → `User factors` × `Singular values` × `Hotel factors ᵀ`  
Reconstructing the full matrix from these factors fills in predicted scores for unvisited hotels.

**Recommendation logic:**  
For a given user → identify unvisited hotels → rank by predicted score from reconstructed matrix → return top-N.

In [ ]:
# Normalise rows so users with very high total spend don't dominate
matrix_norm = normalize(INTERACTION_MATRIX.values, norm='l2')

# Fit SVD with n_components=5 (tuned below)
N_COMPONENTS = 5
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
USER_FACTORS  = svd.fit_transform(matrix_norm)
ITEM_FACTORS  = svd.components_   # shape: (n_components, n_hotels)

print(f'SVD n_components: {N_COMPONENTS}')
print(f'Explained variance ratio: {svd.explained_variance_ratio_}')
print(f'Total variance explained: {svd.explained_variance_ratio_.sum():.3f}')

# Reconstruct predicted rating matrix
RECONSTRUCTED = np.dot(USER_FACTORS, ITEM_FACTORS)
RECON_DF = pd.DataFrame(RECONSTRUCTED, index=INTERACTION_MATRIX.index,
                         columns=INTERACTION_MATRIX.columns)
print('\nReconstructed matrix shape:', RECON_DF.shape)
display(RECON_DF.head())

## 5. Hyperparameter Tuning — n_components

The key hyperparameter for SVD is `n_components` — the number of latent factors. Too few: underfits (misses patterns). Too many: overfits (memorises exact interactions instead of generalising).

We evaluate each value of `n_components` using **reconstruction error** (how well the reduced matrix approximates the original) as the selection criterion.

In [ ]:
from sklearn.metrics import mean_squared_error

components_range = [2, 3, 4, 5, 6, 7, 8]
results_svd = []

for n in components_range:
    svd_tmp = TruncatedSVD(n_components=n, random_state=RANDOM_STATE)
    U = svd_tmp.fit_transform(matrix_norm)
    recon = np.dot(U, svd_tmp.components_)
    rmse = np.sqrt(mean_squared_error(matrix_norm, recon))
    var  = svd_tmp.explained_variance_ratio_.sum()
    results_svd.append({'n_components': n, 'Reconstruction RMSE': round(rmse,5), 'Explained Variance': round(var,4)})

results_df_svd = pd.DataFrame(results_svd)
print('SVD Hyperparameter Search:')
display(results_df_svd)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
results_df_svd.plot(x='n_components', y='Reconstruction RMSE', ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('RMSE vs n_components (lower = better fit)')
results_df_svd.plot(x='n_components', y='Explained Variance', ax=axes[1], marker='o', color='teal')
axes[1].set_title('Explained Variance vs n_components')
axes[1].axhline(0.9, color='red', linestyle='--', label='90% threshold')
axes[1].legend()
plt.tight_layout()
plt.savefig('nb3_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

# Select best: where explained variance first exceeds 0.85 and RMSE stabilises
BEST_N = results_df_svd[results_df_svd['Explained Variance'] >= 0.85]['n_components'].iloc[0]
print(f'\nSelected n_components: {BEST_N} (first to exceed 85% explained variance)')

In [ ]:
# Re-fit with best n_components
SVD_FINAL = TruncatedSVD(n_components=int(BEST_N), random_state=RANDOM_STATE)
USER_FACTORS = SVD_FINAL.fit_transform(matrix_norm)
ITEM_FACTORS = SVD_FINAL.components_
RECON_DF     = pd.DataFrame(np.dot(USER_FACTORS, ITEM_FACTORS),
                              index=INTERACTION_MATRIX.index, columns=INTERACTION_MATRIX.columns)
print(f'Final SVD: n_components={BEST_N}')
print(f'Explained variance: {SVD_FINAL.explained_variance_ratio_.sum():.3f}')

## 6. MLflow Tracking

For recommendation models, we log:
- `n_components` — the tuned SVD hyperparameter
- `explained_variance` — total variance captured
- `reconstruction_rmse` — how well the model approximates the original matrix
- `coverage` — % of all hotels the system can recommend
- `avg_precision_at_3` — of top-3 recommendations, how many did the user actually visit?

In [ ]:
EXPERIMENT_NAME = 'hotel_recommendation'
mlflow.set_experiment(EXPERIMENT_NAME)

# Compute evaluation metrics
from sklearn.metrics import mean_squared_error as mse_fn
recon_rmse = np.sqrt(mse_fn(matrix_norm, np.dot(USER_FACTORS, ITEM_FACTORS)))

# Coverage: % of hotels that appear in at least one recommendation
def get_recommendations(user_code, top_n=3):
    if user_code not in INTERACTION_MATRIX.index: return []
    visited = set(INTERACTION_MATRIX.columns[INTERACTION_MATRIX.loc[user_code] > 0])
    scores  = RECON_DF.loc[user_code]
    unvisited = scores.drop(list(visited), errors='ignore')
    if len(unvisited) == 0: return scores.sort_values(ascending=False).head(top_n).index.tolist()
    return unvisited.sort_values(ascending=False).head(top_n).index.tolist()

all_recs = [h for u in INTERACTION_MATRIX.index for h in get_recommendations(u, 3)]
coverage = len(set(all_recs)) / len(INTERACTION_MATRIX.columns)

# Precision@3: for users with < 9 hotels visited, did recs appear in their actual visits?
partial_users = INTERACTION_MATRIX.index[
    (INTERACTION_MATRIX > 0).sum(axis=1) < 9].tolist()
precisions = []
for u in partial_users[:200]:  # sample for speed
    visited  = set(INTERACTION_MATRIX.columns[INTERACTION_MATRIX.loc[u] > 0])
    recs     = get_recommendations(u, 3)
    # leave-one-out: was any rec in their visited set?
    if len(visited) > 1:
        hit = len(set(recs) & visited)
        precisions.append(hit / 3)
avg_precision_at_3 = np.mean(precisions) if precisions else 0

with mlflow.start_run(run_name=f'svd_n{BEST_N}_components'):
    mlflow.log_params({'n_components': int(BEST_N), 'normalisation': 'l2'})
    mlflow.set_tag('model_type', 'SVD_CollaborativeFiltering')
    mlflow.set_tag('status', 'production')
    metrics = {
        'explained_variance':  round(float(SVD_FINAL.explained_variance_ratio_.sum()), 4),
        'reconstruction_rmse': round(float(recon_rmse), 6),
        'coverage':            round(float(coverage), 4),
        'avg_precision_at_3':  round(float(avg_precision_at_3), 4),
    }
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(SVD_FINAL, 'svd_model')
    rec_run_id = mlflow.active_run().info.run_id

print('MLflow metrics logged:')
for k,v in metrics.items(): print(f'  {k}: {v}')

## 7. Evaluation

Recommender systems cannot use RMSE or accuracy in the same way as regression/classification — there is no single 'correct' recommendation. Instead we use metrics specific to recommendation quality:

| Metric | Formula | What it measures |
|--------|---------|------------------|
| **Reconstruction RMSE** | √(mean((M_original − M_reconstructed)²)) | How accurately SVD reproduces the full interaction matrix |
| **Explained variance** | Σ(singular values²) / Σ(all variance) | % of information in the matrix captured by n_components |
| **Coverage** | unique recommended hotels / total hotels | Are all hotels being recommended, or only popular ones? |
| **Precision@K** | relevant in top-K / K | Of the top-K recommendations, how many did the user actually visit? |

**Why coverage matters:** A recommender that always suggests Hotel K (the most visited) would have low coverage and be useless. Good coverage means the model is surfacing diverse, personalised recommendations.

**Why Precision@3 uses leave-one-out:** We check if any of the top-3 recommendations appear in a user's *actual* visited set. This is an approximation — in a true offline evaluation we would hide some visits during training and check if the model recommends them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Recommendation System Evaluation', fontsize=13, fontweight='bold')

# 1. Explained variance by component
axes[0].bar(range(1, int(BEST_N)+1), SVD_FINAL.explained_variance_ratio_,
            color='steelblue', edgecolor='white')
axes[0].set_title('Variance Explained per Component')
axes[0].set_xlabel('SVD Component'); axes[0].set_ylabel('Explained Variance Ratio')

# 2. Recommendation frequency per hotel (coverage quality)
rec_counts = pd.Series(all_recs).value_counts()
rec_counts.plot(kind='bar', ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title(f'Hotel Recommendation Frequency\nCoverage: {coverage*100:.0f}% of all hotels')
axes[1].set_xlabel('Hotel'); axes[1].tick_params(axis='x', rotation=45)

# 3. Precision@K distribution
if precisions:
    axes[2].hist(precisions, bins=10, color='coral', edgecolor='white')
    axes[2].axvline(avg_precision_at_3, color='red', linestyle='--',
                    label=f'Mean: {avg_precision_at_3:.3f}')
    axes[2].set_title('Precision@3 Distribution')
    axes[2].set_xlabel('Precision@3'); axes[2].legend()

plt.tight_layout()
plt.savefig('nb3_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nModel Summary:')
for k,v in metrics.items(): print(f'  {k}: {v}')

## 9. Save Model & Artefacts

In [ ]:
joblib.dump(SVD_FINAL,           'svd_model.pkl')
joblib.dump(INTERACTION_MATRIX,  'interaction_matrix.pkl')
joblib.dump(RECON_DF,            'reconstructed_matrix.pkl')
joblib.dump(list(INTERACTION_MATRIX.columns), 'hotel_names.pkl')
print('Saved:')
print('  svd_model.pkl              — fitted TruncatedSVD model')
print('  interaction_matrix.pkl     — user × hotel interaction matrix')
print('  reconstructed_matrix.pkl   — predicted scores for all user-hotel pairs')
print('  hotel_names.pkl            — ordered list of hotel names')
print()
# Quick verification
loaded_svd = joblib.load('svd_model.pkl')
print('Loaded SVD n_components:', loaded_svd.n_components)
print('Explained variance:', loaded_svd.explained_variance_ratio_.sum().round(3))

## 8. Recommendation Function

This function is called by both the notebook tests and the Streamlit app. It loads all artefacts from disk — the same files just saved above — ensuring the function behaves identically in Colab, local, and deployed environments.

In [ ]:
def recommend_hotels(user_code, top_n=3, verbose=True):
    """
    Recommend top-N unvisited hotels for a user using the SVD reconstructed matrix.

    Parameters
    ----------
    user_code : int   — userCode from hotels dataset
    top_n     : int   — number of hotels to return
    verbose   : bool  — print explanation

    Returns
    -------
    list of dicts: [{'hotel': str, 'predicted_score': float}]
    """
    matrix  = joblib.load('interaction_matrix.pkl')
    recon   = joblib.load('reconstructed_matrix.pkl')

    if user_code not in matrix.index:
        # Cold start: return globally most popular hotels
        popular = matrix.sum(axis=0).sort_values(ascending=False).head(top_n)
        if verbose:
            print(f'User {user_code} not found — returning most popular hotels (cold start)')
        return [{'hotel': h, 'predicted_score': round(float(s), 2), 'note': 'cold start'}
                for h, s in popular.items()]

    visited   = set(matrix.columns[matrix.loc[user_code] > 0])
    scores    = recon.loc[user_code]
    unvisited = scores.drop(list(visited), errors='ignore')

    if len(unvisited) == 0:
        recs = scores.sort_values(ascending=False).head(top_n)
        note = 'all hotels visited — ranked by preference score'
    else:
        recs = unvisited.sort_values(ascending=False).head(top_n)
        note = f'visited {len(visited)} hotels'

    if verbose:
        print(f'User {user_code} ({note}):')
        for hotel, score in recs.items():
            print(f'  → {hotel}: score={score:.2f}')

    return [{'hotel': h, 'predicted_score': round(float(s), 2)} for h, s in recs.items()]


# ── Test on 3 sample users ──
print('=== Recommendation Tests ===')
partial_users = INTERACTION_MATRIX.index[(INTERACTION_MATRIX > 0).sum(axis=1) < 9][:3]
for u in partial_users:
    print()
    recommend_hotels(u, top_n=3, verbose=True)


## 10. MLflow Run Summary

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
cols = ['tags.mlflow.runName','params.n_components','metrics.explained_variance',
        'metrics.reconstruction_rmse','metrics.coverage','metrics.avg_precision_at_3']
avail = [c for c in cols if c in runs.columns]
display(runs[avail].rename(columns=lambda c: c.replace('tags.','').replace('metrics.','').replace('params.','')))
print('\nTo launch MLflow UI in Colab:')
print('  !pip install pyngrok -q')
print('  from pyngrok import ngrok')
print('  import subprocess; subprocess.Popen(["mlflow","ui","--port","5000"])')
print('  print(ngrok.connect(5000))')